# Quick Start: Gaussian Experiments

This notebook demonstrates OTP-FM (Optimal Transport Potentials for Multi-Marginal Flow Matching) on simple 1D and 2D Gaussian distributions.

In [ ]:
from collections import OrderedDict
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# Import OTP-FM
from otpfm import OTPFM, Curriculum
from otpfm.potentials import W2InfPotential, W2Potential, KLPotential, MMDRBFPotential 

# Import built-in plotting utilities
from experiments.plotting import plot_trajectories_1d, plot_losses

## 1. Define Marginal Distributions

In [ ]:
# Marginal parameters: means and stds for [source, intermediate_1, intermediate_2, target]
means = [0.5, 1.5, 0.4, 0.8]
stds = [0.1, 0.1, 0.1, 0.1]

# Sample from marginals
n_samples = 100_000
dim = 1

xs = [torch.randn(n_samples, dim) * std + mean for mean, std in zip(means, stds)]

# Create dataset and dataloader
# Stack marginals: (n_samples, num_marginals, dim)
xs_stacked = torch.stack(xs, dim=1)
dataset = TensorDataset(xs_stacked)

batch_size = 512
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## 2. Create OTP-FM Model

We define K=2 intermediate potentials at t=0.33 and t=0.67 to enforce the marginal constraints.

In [ ]:
# Define K = 2 intermediate marginal potentials
tks = [0.33, 0.67]  # Intermediate time points
potentials = OrderedDict(
    {
        tks[0]: W2InfPotential(tk=tks[0], strength=500.0, lambda_type="gaussian", width=0.15),
        tks[1]: W2InfPotential(tk=tks[1], strength=500.0, lambda_type="gaussian", width=0.15),
    }
)

# Create model
model = OTPFM(
    d=dim,
    tks=tks,
    potentials=potentials,
    flownet_args={
        'hidden_dim': 256,
        'num_hidden_layers': 2,
        "layernorm": False,
    }
)

print(model)

## 3. Training

The `Curriculum` class manages the OTP alpha schedule, smoothly transitioning from vanilla flow matching (α=0) to full OTP-FM (α=1) during training. The default sigmoid schedule is recommended for stability.

In [ ]:
# Training configuration
n_epochs = 15
total_iterations = n_epochs * len(train_loader)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Create curriculum for OTP alpha scheduling
# This smoothly transitions from vanilla flow matching (alpha=0) to full OTP-FM (alpha=1)
curriculum = Curriculum(total_iterations=total_iterations, schedule="sigmoid")

# Track losses
losses = {
    "train_loss": [],
    "val_loss": [],
    "otp_alpha": [],
}

print("Starting training (takes ~40s on the Apple M4 chip)...")

iteration = 0
for epoch in range(n_epochs):
    model.train()
    epoch_losses = []

    for (batch,) in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        # Get OTP alpha from curriculum
        otp_alpha = curriculum(iteration)

        # Forward pass
        loss = model.forward_with_loss(batch, otp_alpha, do_otp=True)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update EMA
        model.update_ema()

        epoch_losses.append(loss.item())
        iteration += 1

    # Record epoch statistics
    losses["train_loss"].append(np.mean(epoch_losses))
    losses["val_loss"].append(np.mean(epoch_losses))  # Using train as val for this demo
    losses["otp_alpha"].append([epoch, otp_alpha])

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: loss={losses['train_loss'][-1]:.4f}, otp_alpha={otp_alpha:.3f}")

# Plot training losses and OTP alpha schedule
plot_losses(losses, show=True)

## 5. Sample Trajectories

In [ ]:
# Sample trajectories using source samples from the dataset
model.eval()

n_test = 200
# Use source samples from the data (xs[:, 0])
test_x0 = xs_stacked[:n_test, 0]  # (n_test, dim)

with torch.no_grad():
    trajectories, t_eval = model.sample(test_x0, n_steps=20, ema=True)

## 6. Visualize Results

In [ ]:
# Visualize trajectories with marginals
# Normalize initial points relative to source distribution
x0s_normalized = ((test_x0[:, 0] - means[0]) / stds[0]).numpy()

_ = plot_trajectories_1d(
    means=means,
    stds=stds,
    x0s=x0s_normalized,
    t_k=tks,  # Intermediate time points
    xs=trajectories[:, :, 0].T.numpy(),  # (n_samples, n_timesteps)
    t_eval=t_eval.numpy(),
    title="OTP-FM Trajectories",
    n_trajectories=200,
    show=True,
)

## Next Steps

- Try different potential types: `W2Potential`, `KLPotential`, `MMDRBFPotential` and strengths
- Experiment with different lambda functions: `"gaussian"`, `"triangle"`, `"box"` and widths
- See other notebooks for more applications:
  - `02_singlecell_eb.ipynb`: Single-cell trajectory inference
  - `03_gulf_of_mexico.ipynb`: Ocean current modeling
  - `04_beijing_airquality.ipynb`: Air quality forecasting